# Research Navigator for Astronomy: RAG-Based Assistant with NASA ADS

Proyek ini bertujuan mengembangkan sistem Retrieval-Augmented Generation (RAG)
berbasis NASA Astrophysics Data System (ADS) untuk mendukung peneliti, mahasiswa,
serta penggiat astronomi dalam menavigasi dan memahami literatur ilmiah.
Tantangan utama yang dihadapi adalah besarnya jumlah publikasi astronomi yang harus
ditelaah secara manual, yang memakan waktu cukup panjang dan cenderung menghasilkan
interpretasi yang tidak menyeluruh. Sebagai solusi, sistem RAG ini akan memanfaatkan
basis data NASA ADS untuk melakukan pencarian publikasi secara otomatis,
kemudian menghasilkan ringkasan terstruktur serta jawaban atas pertanyaan spesifik.
Dengan demikian, proyek ini diharapkan dapat mempercepat proses literature review,
memfasilitasi eksplorasi topik riset, serta meningkatkan aksesibilitas pengetahuan astronomi
bagi komunitas riset global.

##Tujuan Proyek:

- a) Membangun prototipe sistem Retrieval-Augmented Generation (RAG) yang mampu
melakukan question answering dan summarization berbasis paper dari NASA ADS.

- b) Menyediakan pipeline sederhana untuk querying, retrieval, dan LLM-based generation
agar prototipe dapat dijalankan secara end-to-end.

- c) Mengevaluasi performa awal prototipe melalui:

  • Kualitas ringkasan (perbandingan sederhana dengan abstrak).

  • Relevansi jawaban terhadap pertanyaan uji (evaluasi manual terbatas).

## Data Preparation

In [ ]:
!uv pip install langchain_text_splitters langchain_core sentence-transformers chromadb

In [ ]:
import os
import re
import time
import uuid
import json
import pprint
from datetime import datetime
from urllib.parse import urlencode, quote_plus

import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer, util
import chromadb
from google import genai
from google.genai.errors import ServerError


# ─────────────────────────────────────────────
# Environment Variables
# ─────────────────────────────────────────────
load_dotenv()

NASA_ADS_TOKEN = os.getenv("NASA_ADS_API")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL   = os.getenv("MODEL_GEMINI", "gemini-2.5-flash")

if not NASA_ADS_TOKEN:
    raise EnvironmentError("❌ NASA_ADS_API token not found. Check your .env file.")
if not GEMINI_API_KEY:
    raise EnvironmentError("❌ GEMINI_API_KEY not found. Check your .env file.")

# ─────────────────────────────────────────────
# Model Initialization
# ─────────────────────────────────────────────
_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {_DEVICE.upper()}")

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# ─────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────
ADS_SEARCH_URL  = "https://api.adsabs.harvard.edu/v1/search/query"
ADS_FIELDS      = "bibcode,author,pub,title,year,keyword,abstract"
ADS_YEAR_RANGE  = "2010 TO 2025"
LIST_COLUMNS    = ("author", "title", "keyword")   # fields that come as lists from ADS
REQUIRED_COLS   = ("bibcode", "abstract")          # rows missing these are dropped
REQUEST_DELAY   = 1.0                              # seconds between topic requests (rate-limit)

# ─────────────────────────────────────────────
# Topics
# ─────────────────────────────────────────────
ASTRONOMY_TOPICS: dict[str, str] = {
    "cosmology": (
        '"cosmology" OR "dark matter" OR "dark energy" OR "cosmic microwave background" '
        'OR "large scale structure" OR "galaxy cluster" OR "baryon acoustic oscillations" '
        'OR "gravitational lensing" OR "reionization" OR "cosmic web" OR "redshift survey" '
        'OR "AGN feedback" OR "galaxy merger" OR "supermassive black hole"'
    ),
    "stellar_physics": (
        '"stellar evolution" OR "stellar structure" OR "star formation" OR "supernova" '
        'OR "novae" OR "white dwarf" OR "neutron star" OR "stellar winds" '
        'OR "magnetic field" OR "variable stars" OR "binary stars" OR "stellar nucleosynthesis" '
        'OR "massive stars" OR "Hertzsprung-Russell diagram"'
    ),
    "solar_system": (
        '"solar system" OR "sun" OR "heliosphere" OR "solar wind" OR "solar flare" '
        'OR "coronal mass ejection" OR "magnetosphere" OR "planets" OR "moons" '
        'OR "asteroids" OR "comets" OR "Kuiper Belt" OR "planetary atmospheres" '
        'OR "exoplanets" OR "planetary formation" OR "space weather"'
    ),
    "galactic_astronomy": (
        '"Milky Way" OR "galactic structure" OR "galactic dynamics" OR "interstellar medium" '
        'OR "star clusters" OR "molecular clouds" OR "star-forming regions" '
        'OR "spiral arms" OR "galactic halo" OR "chemical evolution" '
        'OR "stellar populations" OR "Gaia mission"'
    ),
    "high_energy": (
        '"X-ray astronomy" OR "gamma-ray" OR "black hole" OR "neutron star" '
        'OR "pulsar" OR "magnetar" OR "accretion disk" OR "relativistic jets" '
        'OR "supernova remnant" OR "cosmic ray" OR "compact object" OR "gravitational wave" '
        'OR "multi-messenger astronomy" OR "high-energy astrophysics"'
    ),
}

# ─────────────────────────────────────────────
# Core Fetch Function
# ─────────────────────────────────────────────
def get_ads_papers(
    token: str,
    body: str = "x-ray astronomy",
    rows: int = 100,
) -> pd.DataFrame:
    """
    Fetch peer-reviewed academic papers from NASA ADS API for a given research topic.

    Parameters
    ----------
    token : str
        NASA ADS API access token.
    body : str, optional
        Search keyword or topic within the paper body (default: "x-ray astronomy").
    rows : int, optional
        Number of papers to retrieve (default: 100).

    Returns
    -------
    pd.DataFrame
        Clean DataFrame with columns:
        ['author', 'title', 'pub', 'bibcode', 'year', 'keyword', 'abstract']
    """
    params = {
        "q":    f"body:({body}) AND year:[{ADS_YEAR_RANGE}]",
        "fl":   ADS_FIELDS,
        "rows": rows,
        "sort": "score desc",
        "fq":   "property:refereed",
    }
    url = f"{ADS_SEARCH_URL}?{urlencode(params)}"

    response = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=30)
    response.raise_for_status()

    docs = response.json().get("response", {}).get("docs", [])
    if not docs:
        print("  ⚠️  No results found.")
        return pd.DataFrame()

    df = pd.DataFrame(docs)

    # Keep only known columns that are present
    keep = [c for c in ADS_FIELDS.split(",") if c in df.columns]
    df = df[keep]

    # Convert list fields → comma-separated strings
    for col in LIST_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: ", ".join(x) if isinstance(x, list) else (x or "")
            )

    # Drop rows missing critical fields
    df.dropna(subset=[c for c in REQUIRED_COLS if c in df.columns], inplace=True)

    return df

# ─────────────────────────────────────────────
# Multi-topic Collector
# ─────────────────────────────────────────────
def collect_astronomy_data(
    token: str,
    topics: dict[str, str] = ASTRONOMY_TOPICS,
    rows_per_topic: int = 100,
) -> pd.DataFrame:
    """
    Collects astronomy research papers from NASA ADS for given topics.

    Parameters
    ----------
    token : str
        NASA ADS API token.
    topics : dict
        Dictionary of topics with {topic_name: query_body}.
        Example:
            {
                "galaxy": '"galaxy" OR "galaxies" OR "extragalactic"',
                "stellar": '"stellar evolution" OR "supernova"',
            }
    rows_per_topic : int, optional
        Number of papers per topic (default = 100).

    Returns
    -------
    pd.DataFrame
        Combined DataFrame with columns [author, title, pub, bibcode, keyword, abstract, topic].
    """
    all_frames: list[pd.DataFrame] = []

    for topic, body_query in tqdm(topics.items(), desc="📡 Fetching topics", unit="topic"):
        print(f"  → {topic}")
        try:
            df = get_ads_papers(token, body=body_query, rows=rows_per_topic)
            if not df.empty:
                df["topic"] = topic
                all_frames.append(df)
        except requests.HTTPError as e:
            print(f"  ❌ HTTP error for topic '{topic}': {e}")
        except Exception as e:
            print(f"  ❌ Unexpected error for topic '{topic}': {e}")

        time.sleep(REQUEST_DELAY)   # respect NASA ADS rate limit

    if not all_frames:
        print("⚠️  No data collected from any topic.")
        return pd.DataFrame()

    df_combined = (
        pd.concat(all_frames, ignore_index=True)
        .drop_duplicates(subset=["bibcode"])
        .reset_index(drop=True)
    )

    print(f"\n✅ Collected {len(df_combined)} unique papers across {len(topics)} topics.")
    return df_combined

In [ ]:
token = os.getenv("NASA_ADS_API")

df_data = collect_astronomy_data(token, rows_per_topic=1000)
print(df_data.head())

In [ ]:
df_data.sample(20)

In [ ]:
filename = f"astronomy_dataset_{len(df_data)}rows_{datetime.now().strftime('%Y%m%d')}.csv"
df_data.to_csv(filename, index=False, encoding="utf-8")

print(f"✅ File berhasil disimpan sebagai: {filename}")

## Text Preparation

Tujuan utama

Buat RAG kita butuh teks yang bersih, representatif, dan ter-segmentasi jadi potongan (chunks) yang mudah di-embed dan di-retrieve. Selain teks itu sendiri, harus ada metadata (sumber, bibcode, topic, year, dll) supaya setiap jawaban bisa ditelusuri asalnya.



In [ ]:
# Pre-compile regex patterns once (avoid recompiling on every row)
_RE_WHITESPACE  = re.compile(r'\s+')
_RE_BRACKETS    = re.compile(r'[{}[\]<>]')
_RE_BACKSLASH   = re.compile(r'\\+')
_RE_DASHES      = re.compile(r'[–—]')


def clean_text(text: str) -> str:
    """Clean and normalize a single text string."""
    if not isinstance(text, str):
        return ""
    text = _RE_WHITESPACE.sub(' ', text)
    text = _RE_BRACKETS.sub('', text)
    text = _RE_BACKSLASH.sub('', text)
    text = _RE_DASHES.sub('-', text)
    return text.strip()


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and prepare the astronomy dataset for embedding.

    Steps:
    - Combine title, abstract, and keywords into one 'content' column.
    - Remove extra whitespace and unwanted characters.
    - Drop duplicates and empty rows.
    """
    df = df.copy()

    # Safely get columns, fallback to empty string Series
    title    = df['title'].fillna('')    if 'title'    in df.columns else pd.Series('', index=df.index)
    abstract = df['abstract'].fillna('') if 'abstract' in df.columns else pd.Series('', index=df.index)
    keyword  = df['keyword'].fillna('')  if 'keyword'  in df.columns else pd.Series('', index=df.index)

    # Combine main text fields (vectorised — no .apply loop)
    df['content'] = title + '. ' + abstract + ' ' + keyword

    # Clean all text columns at once
    for col in ('title', 'abstract', 'keyword', 'content'):
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

    # Drop empty or duplicated content
    mask = df['content'].str.strip() != ''
    df = df[mask].drop_duplicates(subset=['content'])

    # Lowercase for semantic uniformity (vectorised)
    df['content'] = df['content'].str.lower()

    df.reset_index(drop=True, inplace=True)
    return df

In [ ]:
df = df_data[['bibcode', 'title', 'abstract', 'keyword', 'pub', 'year', 'topic']].copy()

df = clean_dataframe(df)

df = df[df['content'].str.len() > 100]
df.info()

## Chunking

Chunking adalah proses memecah teks panjang menjadi potongan-potongan kecil agar mudah diproses oleh model AI. Tujuannya menjaga konteks, menghindari pemotongan makna, dan meningkatkan akurasi embedding serta pencarian informasi.

In [ ]:
# Metadata columns to carry forward — centralised so it's easy to extend
_METADATA_COLS = ("bibcode", "title", "year", "pub", "keyword", "topic")

# Fallback column if 'content' is missing
_FALLBACK_COL = "abstract"


def chunk_from_content(
    df: pd.DataFrame,
    text_col: str = "content",
    chunk_size: int = 800,
    chunk_overlap: int = 150,
) -> list[Document]:
    """
    Split long text (e.g. content or abstract) into smaller chunks for RAG/embedding.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing a text column (default: 'content').
    text_col : str, optional
        Column name containing text to split (default: 'content').
    chunk_size : int, optional
        Max characters per chunk (default: 800).
    chunk_overlap : int, optional
        Overlap between chunks (default: 150).

    Returns
    -------
    list[Document]
        List of LangChain Document objects containing chunked text and metadata.
    """
    if df.empty:
        raise ValueError("Input DataFrame is empty.")

    # ── Column resolution ────────────────────────────────────────────────
    if text_col not in df.columns:
        if _FALLBACK_COL in df.columns:
            print(f"⚠️  Column '{text_col}' not found. Using '{_FALLBACK_COL}' instead.")
            text_col = _FALLBACK_COL
        else:
            raise ValueError(
                f"DataFrame has neither '{text_col}' nor '{_FALLBACK_COL}' column. "
                f"Available columns: {list(df.columns)}"
            )

    # ── Build splitter once ──────────────────────────────────────────────
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )

    # ── Pre-resolve which metadata columns actually exist ────────────────
    active_meta_cols = [c for c in _METADATA_COLS if c in df.columns]

    # ── Collect texts and metadata in one pass ───────────────────────────
    texts: list[str] = []
    metadatas: list[dict] = []

    for row in df[active_meta_cols + [text_col]].itertuples(index=False):
        text = str(getattr(row, text_col)).strip()
        if not text or text == "nan":   # ← fix: skip "nan" string from pd.NA
            continue
        texts.append(text)
        metadatas.append({
            col: (str(getattr(row, col)) if getattr(row, col) is not None else "")
            for col in active_meta_cols   # ← fix: ensure all values are str for ChromaDB
        })

    if not texts:
        raise ValueError(f"No valid text found in column '{text_col}' after filtering.")

    # ── Single splitter call for all documents ───────────────────────────
    docs = splitter.create_documents(texts, metadatas=metadatas)

    print(
        f"✅ Created {len(docs)} chunks from {len(texts)} non-empty documents "
        f"(source: '{text_col}', chunk_size={chunk_size}, overlap={chunk_overlap})."
    )
    return docs

In [ ]:
docs = chunk_from_content(df, text_col="content", chunk_size=900, chunk_overlap=150)
print(docs[0].page_content[:300])
print(docs[0].metadata)

In [ ]:
print(docs[-1].page_content[:300])
print(docs[-1].metadata)

## Embedding (Representasi vektor)

Embedding adalah proses mengubah teks menjadi representasi vektor numerik agar komputer dapat memahami maknanya. Tujuannya untuk memudahkan pencarian semantik, clustering, dan analisis kesamaan antar teks secara efisien dalam sistem berbasis AI atau machine learning.

In [ ]:
def embed_chunks(
    chunks: list[Document],
    model: SentenceTransformer,
    batch_size: int = 64,
) -> list[Document]:
    """
    Embed LangChain Document chunks using SentenceTransformer.
    Each Document's metadata will contain its corresponding embedding vector.

    Parameters
    ----------
    chunks : list[Document]
        List of LangChain Document objects. Each Document should have:
            - page_content : str  -> the text to be embedded
            - metadata : dict     -> metadata container (will be updated with 'embedding')
    model : SentenceTransformer
        A pre-loaded SentenceTransformer model instance (e.g. embed_model from config.py).
        Recommended models for RAG:
            - 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1' (default, fast + Q&A optimised)
            - 'intfloat/e5-base-v2' (balanced, general-purpose)
            - 'BAAI/bge-large-en-v1.5' (high-quality semantic embeddings)
            - 'nomic-ai/nomic-embed-text-v1.5' (latest open model)
    batch_size : int, optional
        Number of text chunks processed per batch to balance speed and memory usage.

    Returns
    -------
    list[Document]
        The same list of chunks, with each Document's metadata containing:
            metadata["embedding"] : np.ndarray (vector representation of the text)

    Notes
    -----
    • This function performs local embedding generation (no API calls).
    • Embeddings are normalized to improve cosine similarity consistency.
    • GPU acceleration will be used automatically if available.
    • Large embedding vectors can consume memory; for large datasets, consider
      storing embeddings separately instead of inside metadata.
    """
    if not chunks:
        print("⚠️  No chunks to embed.")
        return chunks

    embed_dim = model.get_sentence_embedding_dimension()

    # ── Filter out empty chunks before encoding ──────────────────────────
    valid_indices = [i for i, doc in enumerate(chunks) if doc.page_content.strip()]
    skipped = len(chunks) - len(valid_indices)
    if skipped:
        print(f"⚠️  Skipping {skipped} empty chunks.")

    valid_texts = [chunks[i].page_content for i in valid_indices]
    n = len(valid_texts)

    if n == 0:
        print("⚠️  All chunks are empty. Nothing embedded.")
        return chunks

    # ── Pre-allocate output array ────────────────────────────────────────
    all_embeddings = np.zeros((n, embed_dim), dtype=np.float32)
    failed_batches: list[int] = []

    for batch_idx, start in enumerate(tqdm(
        range(0, n, batch_size),
        desc=f"🔢 Embedding {n} chunks",
        unit="batch",
    )):
        batch = valid_texts[start : start + batch_size]
        try:
            all_embeddings[start : start + len(batch)] = model.encode(
                batch,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=True,  # unit-length vectors for cosine similarity
            )
        except Exception as e:
            # Zero-vector fallback already in place from np.zeros pre-allocation
            failed_batches.append(batch_idx)
            print(f"⚠️  Batch {batch_idx} (index {start}) failed: {e}")

    if failed_batches:
        print(f"⚠️  {len(failed_batches)} batch(es) failed and were zero-filled: {failed_batches}")

    # ── Attach embeddings only to valid (non-empty) chunks ───────────────
    for rank, doc_idx in enumerate(valid_indices):
        chunks[doc_idx].metadata["embedding"] = all_embeddings[rank]

    model_name = getattr(model, "_model_card_data", {}).get("model_name", str(model.__class__.__name__))
    print(f"✅ Embedded {n}/{len(chunks)} chunks (dim={embed_dim}).")
    return chunks

In [ ]:
def get_embed_model(
    model_name: str = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
) -> SentenceTransformer:
    """Load SentenceTransformer model onto the best available device."""
    print(f"🔧 Loading model '{model_name}' on {_DEVICE.upper()}...")
    return SentenceTransformer(model_name, device=_DEVICE)

embed_model = get_embed_model()

In [ ]:
embedded_docs = embed_chunks(docs, model=embed_model)

# Cek hasil
print(embedded_docs[0].metadata.keys())
print(len(embedded_docs[0].metadata["embedding"]))

## Vector Storage

Vector storage adalah tempat menyimpan representasi vektor hasil embedding. Tujuannya agar sistem dapat melakukan pencarian semantik cepat, menemukan kemiripan antar dokumen, dan mendukung aplikasi seperti chatbot RAG atau rekomendasi berbasis konteks.

In [ ]:
# ── Singleton ChromaDB client cache ─────────────────────────────────────────
_chroma_clients: dict[str, chromadb.PersistentClient] = {}


def _get_chroma_client(persist_path: str) -> chromadb.PersistentClient:
    """Return a cached ChromaDB client for the given path."""
    if persist_path not in _chroma_clients:
        _chroma_clients[persist_path] = chromadb.PersistentClient(path=persist_path)
    return _chroma_clients[persist_path]


def _clean_metadata(meta: dict) -> dict:
    """
    Remove non-serializable values from metadata.
    ChromaDB only accepts str, int, float, or bool values.
    """
    clean = {}
    for k, v in meta.items():
        if k == "embedding":
            continue  # always strip embedding vector from metadata
        if isinstance(v, np.ndarray):
            continue  # skip any leftover array
        if isinstance(v, (str, int, float, bool)):
            clean[k] = v
        elif v is None:
            clean[k] = ""  # ChromaDB rejects None — use empty string
        else:
            clean[k] = str(v)  # safe fallback for unexpected types
    return clean


def store_to_chroma(
    chunks: list[Document],
    collection_name: str = "astro_paper",
    persist_path: str = "./chroma_db",
    batch_size: int = 1000,
) -> chromadb.Collection:
    """
    Store embedded LangChain Document chunks into a persistent ChromaDB collection.

    Parameters
    ----------
    chunks : list[Document]
        List of LangChain Document objects containing:
            - page_content (str): text content
            - metadata["embedding"] (np.ndarray): vector representation of the text
            - other metadata fields (source, title, etc.)
    collection_name : str, optional
        Name of the ChromaDB collection to create or append to.
        Default = "astro_paper"
    persist_path : str, optional
        Directory path where ChromaDB will store persistent data.
        Default = "./chroma_db"
    batch_size : int, optional
        Number of documents to insert per batch (helps with memory efficiency).
        Default = 1000

    Returns
    -------
    chromadb.Collection
        The ChromaDB collection object after insertion.

    Notes
    -----
    • Embeddings stored as lists (not numpy arrays) to avoid serialization errors.
    • Numpy arrays in metadata are automatically removed.
    • Uses batched inserts with tqdm progress bar for large datasets.
    • ChromaDB client is cached — safe to call multiple times without reconnecting.
    • None metadata values are converted to empty strings for ChromaDB compatibility.
    """
    if not chunks:
        raise ValueError("No chunks provided to store_to_chroma.")

    # ── Init client & collection ─────────────────────────────────────────────
    client = _get_chroma_client(persist_path)
    collection = client.get_or_create_collection(name=collection_name)
    print(f"✅ Collection '{collection_name}' ready at '{persist_path}'")

    # ── Build insertion lists in one pass ────────────────────────────────────
    ids, embeddings, documents, metadatas = [], [], [], []

    for doc in chunks:
        emb = doc.metadata.get("embedding")
        if emb is None:
            continue  # skip chunks that were not embedded

        ids.append(str(uuid.uuid4()))          # UUID avoids ID collision on re-runs
        embeddings.append(emb.tolist())        # np.ndarray → list for serialization
        documents.append(doc.page_content)
        metadatas.append(_clean_metadata(doc.metadata))

    total = len(ids)
    if total == 0:
        print("⚠️  No valid embedded chunks found. Nothing stored.")
        return collection

    print(f"📦 Inserting {total} documents into '{collection_name}'...")

    # ── Batched insert ───────────────────────────────────────────────────────
    for start in tqdm(
        range(0, total, batch_size),
        desc="💾 Storing to ChromaDB",
        unit="batch",
    ):
        end = start + batch_size
        collection.add(
            ids=ids[start:end],
            embeddings=embeddings[start:end],
            documents=documents[start:end],
            metadatas=metadatas[start:end],
        )

    print(f"✅ Stored {total} documents into collection '{collection_name}'.")
    return collection

In [ ]:
collection = store_to_chroma(
    chunks=embedded_docs,
    collection_name="astro_paper",
    persist_path="./chroma_db",
    batch_size=1000
)

## Query

Query adalah proses pencarian informasi di dalam database vektor dengan menggunakan teks sebagai input. Tujuannya untuk menemukan potongan teks yang paling relevan atau mirip secara makna dengan pertanyaan pengguna berdasarkan perbandingan vektor embedding.



In [ ]:
# ── Connect to existing ChromaDB collection ───────────────────────────────────
persist_path    = "./chroma_db"
collection_name = "astro_paper"

client     = chromadb.PersistentClient(path=persist_path)
collection = client.get_collection(name=collection_name)
print(f"✅ Connected to collection '{collection_name}' — {collection.count()} documents.")


In [ ]:
# ── Query parameters ──────────────────────────────────────────────────────────
query_text     = "How are cosmological parameters measured using galaxy clusters?"
filter_keyword = "cosmology"   # set None to disable filtering
n_results      = 10
score_threshold = 0.3          # drop results below this similarity

# ── Embed query with the same model used at ingestion ────────────────────────
query_embedding = embed_model.encode(
    [query_text],
    normalize_embeddings=True,
    convert_to_numpy=True,
).tolist()

# ── Retrieve from ChromaDB ────────────────────────────────────────────────────
raw = collection.query(
    query_embeddings=query_embedding,
    n_results=n_results,
    include=["documents", "metadatas", "distances"],
)

docs      = raw["documents"][0]
metas     = raw["metadatas"][0]
distances = raw["distances"][0]

# ── Filter & format results ───────────────────────────────────────────────────
fk = filter_keyword.lower() if filter_keyword else None
formatted_results = []

for doc, meta, dist in zip(docs, metas, distances):
    score = round(1.0 - dist, 4)   # cosine distance → similarity

    if score < score_threshold:
        continue

    if fk:
        topic   = meta.get("topic",   "").lower()
        keyword = meta.get("keyword", "").lower()
        if fk not in topic and fk not in keyword:
            continue

    formatted_results.append({
        "score":   score,
        "title":   meta.get("title",   "N/A"),
        "source":  meta.get("bibcode", "N/A"),
        "topic":   meta.get("topic",   "N/A"),
        "keyword": meta.get("keyword", "N/A"),
        "abstract": doc[:500] + ("..." if len(doc) > 500 else ""),
    })

# Results already sorted by ChromaDB; re-sort after filtering
formatted_results.sort(key=lambda x: x["score"], reverse=True)

# ── Display ───────────────────────────────────────────────────────────────────
if formatted_results:
    label = f" (filtered by '{filter_keyword}')" if filter_keyword else ""
    print(f"\n🎯 {len(formatted_results)} result(s) found{label}:\n")

    for i, r in enumerate(formatted_results, 1):
        print(f"🔹 [{i}] {r['title']}")
        print(f"   Score   : {r['score']:.4f}  |  Topic: {r['topic']}")
        print(f"   Bibcode : {r['source']}")
        print(f"   Keywords: {r['keyword']}")
        print(f"   Preview : {r['abstract']}\n")
else:
    hint = f" with keyword '{filter_keyword}'" if filter_keyword else ""
    print(f"⚠️  No results found{hint}. Try lowering score_threshold or changing the keyword.")

# RAG (Retrieval-Augmented Generation)

RAG adalah metode yang menggabungkan pencarian informasi (retrieval) dari basis data vektor dengan kemampuan generatif model bahasa. Tujuannya agar model tidak hanya “menebak” jawaban dari pengetahuan internalnya, tetapi juga menggunakan data eksternal yang relevan, akurat, dan kontekstual untuk menghasilkan respons yang lebih faktual dan dapat dipertanggungjawabkan.

RAG digunakan untuk menjawab pertanyaan berbasis dokumen, meringkas teks, atau membuat chatbot yang bisa menarik data dari koleksi artikel, laporan, atau database ilmiah secara otomatis.

In [ ]:
# ─────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────
_SYSTEM_INSTRUCTION = (
    "You are AstroRAG, an expert scientific AI assistant specialising in astronomy, "
    "astrophysics, and cosmology. You answer questions with precision and clarity, "
    "strictly grounded in the provided paper abstracts. "
    "You never fabricate facts, invent citations, or speculate beyond what the abstracts state. "
    "If the abstracts do not contain enough information to answer, say so explicitly."
)

_PROMPT_TEMPLATE = """\
## ROLE
You are AstroRAG — a scientific AI assistant specialising in astronomy, astrophysics, and cosmology.
Your sole knowledge source for this answer is the numbered paper abstracts provided below.

## STRICT RULES
1. ONLY use information explicitly stated in the abstracts. Do not use prior knowledge.
2. ALWAYS cite the paper number(s) supporting each claim, e.g. [1] or [2,4].
3. If no abstract contains sufficient information, respond EXACTLY:
   "The available abstracts do not contain enough information to answer this question."
4. Do NOT speculate, infer beyond the text, or fill gaps with general knowledge.
5. Do NOT repeat the question or the abstracts verbatim in your answer.

## ANSWER FORMAT
Structure your response as follows:

**Direct Answer** (8–10 sentences summarising the core answer)

**Supporting Evidence** (cite abstracts inline, e.g. [1], [2,4])
- Explain the key findings from the papers that support your answer.
- Define technical terms on first use (e.g., CMB, BAO, AGN feedback).

**Caveats & Open Questions** (only if relevant)
- Note limitations, conflicting findings, or areas still under investigation.

---

## RETRIEVED PAPER ABSTRACTS
{context_string}

---

## USER QUESTION
{query_text}

## YOUR ANSWER
"""


# ─────────────────────────────────────────────
# Retrieve & Augment
# ─────────────────────────────────────────────
def retrieve_and_augment_prompt(
    query_text: str,
    collection: chromadb.Collection,
    n_results: int = 5,
) -> tuple[str, list[tuple[str, str, str]]]:
    """
    Retrieve relevant astronomy paper abstracts from a ChromaDB collection,
    then construct a context-enriched prompt for an astronomy-focused RAG system.

    Args:
        query_text (str): The user's natural language question.
        collection (ChromaDB Collection): Vector DB collection containing paper abstracts.
        n_results (int): Number of most relevant chunks to retrieve.

    Returns:
        tuple:
            - str: Constructed, context-enriched prompt ready for LLM.
            - list: List of tuples (bibcode, title, pub) for retrieved papers.
    """
    print(f"\nStep 1: Received user query -> '{query_text}'")

    # Embed query using the shared embed_model
    query_embedding = embed_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).tolist()

    # Retrieve from ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    retrieved_chunks = results["documents"][0]
    metadata         = results["metadatas"][0]
    print(f"Step 2: Retrieved {len(retrieved_chunks)} relevant abstracts.")

    # Build context string + paper info (sorted newest first)
    papers = sorted(
        zip(retrieved_chunks, metadata),
        key=lambda x: int(x[1].get("year", 0) or 0),  # handle None, "", "0"
        reverse=True,
    )

    context_string = ""
    paper_info: list[tuple[str, str, str]] = []

    for i, (doc, meta) in enumerate(papers, 1):
        bibcode = meta.get("bibcode", "N/A")
        title   = meta.get("title",   "Unknown Title")
        pub     = meta.get("pub",     "Unknown Journal")
        paper_info.append((bibcode, title, pub))
        context_string += (
            f"[{i}] {title} ({meta.get('year', 'Unknown')}) — {pub}\n"
            f"Bibcode: {bibcode}\n"
            f"{doc}\n\n---\n\n"
        )

    prompt = _PROMPT_TEMPLATE.format(
        context_string=context_string,
        query_text=query_text,
    ).strip()

    print("Step 3: Context-enriched prompt successfully generated.")
    return prompt, paper_info


# ─────────────────────────────────────────────
# Full RAG Pipeline
# ─────────────────────────────────────────────
def generate_answer_with_rag(
    query_text: str,
    collection: chromadb.Collection,
    n_results: int = 5,
    model_name: str = GEMINI_MODEL,
) -> str:
    """
    Full RAG pipeline: retrieve -> augment -> generate answer with sources.

    Args:
        query_text (str): User question.
        collection (ChromaDB Collection): Collection of vectorized abstracts.
        n_results (int): Number of top documents to retrieve.
        model_name (str): Gemini model name from config.py.

    Returns:
        str: LLM answer with appended list of retrieved papers.
    """
    # 1. Retrieve + build prompt
    prompt, paper_info = retrieve_and_augment_prompt(query_text, collection, n_results)

    # 2. Send to Gemini via shared gemini_client from config.py
    print(f"\nStep 4: Sending augmented prompt to {model_name}...\n")
    try:
        response = gemini_client.models.generate_content(
            model=model_name,
            contents=prompt,
            config={
                "system_instruction": _SYSTEM_INSTRUCTION,
                "temperature": 0.15,  # low = factual, no hallucination
            },
        )
        answer = response.text.strip()
    except ServerError as e:
        print(f"❌ Gemini API error: {e}")
        return "An error occurred while generating the answer. Please try again."

    print("Step 5: LLM response received.\n")

    # 3. Append sources
    sources = "\n".join(
        f"[{i}] {title} — {pub} | {bibcode}"
        for i, (bibcode, title, pub) in enumerate(paper_info, 1)
    )
    return f"{answer}\n\n=== SOURCES USED ===\n{sources}"

In [ ]:
query_example = "What are the latest findings about dark energy and cosmic acceleration?"
final_answer = generate_answer_with_rag(query_example, collection, n_results=10)

print(final_answer)

In [ ]:
query_example = "What are the latest findings about dark energy and cosmic acceleration?"
final_answer = generate_answer_with_rag(query_example, collection, n_results=10)

print(final_answer)

## Testing Answear

In [ ]:
test_questions = [
    "What is the current scientific understanding of dark energy and its role in the universe’s accelerated expansion?",
    "What is the significance of the Cosmic Microwave Background in cosmology?",
    "How are exoplanets detected and what methods are most successful today?",
    "What are gravitational waves and how were they first detected?",
    "What evidence supports the existence of dark matter?"
]

gold_answers = [
    "Dark energy is an unknown form of energy responsible for the universe’s accelerated expansion, comprising about 68% of total energy. It was inferred from Type Ia supernova observations and represented as the cosmological constant in the ΛCDM model, though its physical nature remains unclear.",
    "The Cosmic Microwave Background is relic radiation from the early universe, emitted about 380,000 years after the Big Bang. It provides temperature and density fluctuations used to measure cosmological parameters like age, composition, and geometry of the universe.",
    "Exoplanets are detected mainly via the transit and radial velocity methods. The transit method measures dips in starlight as planets pass in front, while the radial velocity method observes Doppler shifts. Missions like Kepler and TESS have used these methods to discover thousands of exoplanets.",
    "Gravitational waves are ripples in spacetime predicted by Einstein’s General Relativity, first directly detected by LIGO in 2015 from merging black holes. Their detection confirmed a major prediction of relativity and opened a new field of gravitational-wave astronomy.",
    "Dark matter evidence comes from galaxy rotation curves, gravitational lensing, and cosmic microwave background anisotropies. It interacts gravitationally but not electromagnetically, accounting for about 27% of the universe’s total mass-energy content."
]

In [ ]:
def evaluate_rag_answers(
    model_answers: list[str],
    gold_answers: list[str],
    eval_model = embed_model,   # default ke embed_model dari config
) -> tuple[list[float], float]:
    """
    Evaluate similarity between model-generated and reference answers
    using cosine similarity via SentenceTransformer.
    """
    if len(model_answers) != len(gold_answers):
        raise ValueError(
            f"Length mismatch: {len(model_answers)} model answers "
            f"vs {len(gold_answers)} gold answers."
        )

    scores = []
    for i, (pred, gold) in enumerate(zip(model_answers, gold_answers), 1):
        emb_pred = eval_model.encode(pred, convert_to_tensor=True, normalize_embeddings=True)
        emb_gold = eval_model.encode(gold, convert_to_tensor=True, normalize_embeddings=True)
        similarity = util.cos_sim(emb_pred, emb_gold).item()
        scores.append(similarity)
        print(f"Q{i}: Similarity = {similarity:.3f}")

    avg_score = float(np.mean(scores))
    print(f"\n📊 Average Similarity Score: {avg_score:.3f}")
    return scores, avg_score

In [ ]:
# === Jalankan Semua Pertanyaan Uji ===
print("\n=== RUNNING RAG TEST SUITE ===")
rag_generated_answers = []
for q in test_questions:
    ans = generate_answer_with_rag(q, collection, n_results=5)
    rag_generated_answers.append(ans)
    print("\n--- FINAL ANSWER ---")
    print(ans)
    print("\n" + "=" * 20 + "\n")

In [ ]:
# === Evaluasi Jawaban ===
print("\n=== EVALUATION RESULTS ===")
scores, avg = evaluate_rag_answers(rag_generated_answers, gold_answers)